<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/5_1_Nonlinear_Nonparametric_Models_Decision_Trees_(CART).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part V. Nonlinear Nonparametric Models.
## 1. Decision Trees (CART)

## 2. Bagging and Random Forest

## 3. Gradient Boosting (XGBoost, LightGBM, CatBoost)


## 5.1. Решающие деревья (CART): теория, критерии расщепления и вероятностный смысл

Линейные модели накладывают жёсткую параметрическую форму на зависимость. Решающие деревья предлагают принципиально иной подход — они рекурсивно разбивают пространство признаков на подобласти, в каждой из которых предсказание просто. Эта идея приводит к кусочно‑постоянным функциям, способным улавливать сложные нелинейные закономерности без предположений о глобальной структуре.

### 1. Геометрическая и функциональная интерпретация

Решающее дерево задаёт разбиение входного пространства $\mathbb{R}^D$ на множество непересекающихся гиперпрямоугольников, грани которых параллельны осям координат. Каждая такая область $R_m \subseteq \mathbb{R}^D$ ассоциирована с листом дерева, и модель предсказывает постоянное значение $c_m$ для всех наблюдений, попавших в $R_m$:

$$
f(\mathbf{x}) = \sum_{m=1}^{M} c_m \cdot \mathbf{1}_{\{\mathbf{x} \in R_m\}},
$$

где $M$ — число листьев, а $\mathbf{1}_{\{\cdot\}}$ — индикаторная функция. Таким образом, дерево реализует кусочно‑постоянную аппроксимацию неизвестной целевой зависимости.

Структура разбиения задаётся **рекурсивным бинарным деревом**: каждый внутренний узел содержит условие вида «$x_j \le \theta$», где $x_j$ — один из признаков, а $\theta$ — пороговое значение. Объекты, удовлетворяющие условию, направляются в левое поддерево, остальные — в правое. Процесс рекурсивно продолжается до тех пор, пока не сработает критерий остановки, после чего узел становится листом. Геометрически каждое расщепление проводит границу, параллельную координатным осям, поэтому результирующие области являются объединением многомерных прямоугольников со сторонами, параллельными осям. Это свойство одновременно обеспечивает высокую интерпретируемость: любое решение дерева можно представить как набор правил «если … то …».

### 2. Жадное построение дерева для регрессии

Пусть обучающая выборка состоит из пар $(\mathbf{x}_i, y_i) \in \mathbb{R}^D \times \mathbb{R}$, $i = 1,\dots,n$. В узле $t$, содержащем $m$ наблюдений, ищется расщепление по признаку $j$ и порогу $\theta$, минимизирующее сумму квадратов ошибок в двух образовавшихся дочерних узлах:

$$
Q(j, \theta) = \min_{c_1} \sum_{i \in R_1(j,\theta)} (y_i - c_1)^2 + \min_{c_2} \sum_{i \in R_2(j,\theta)} (y_i - c_2)^2,
$$

где
$$
R_1(j,\theta) = \{i \mid x_{ij} \le \theta\}, \qquad R_2(j,\theta) = \{i \mid x_{ij} > \theta\}.
$$

Оптимальные значения констант $c_1$ и $c_2$, минимизирующие каждое из слагаемых, суть выборочные средние целевой переменной в соответствующих регионах:

$$
c_1 = \frac{1}{m_L} \sum_{i \in R_1} y_i, \qquad c_2 = \frac{1}{m_R} \sum_{i \in R_2} y_i,
$$

где $m_L = |R_1|$, $m_R = |R_2|$, $m_L + m_R = m$. После подстановки этих значений критерий приобретает эквивалентную форму — максимизацию **уменьшения дисперсии**:

$$
\Delta = \sigma^2_{\text{parent}} - \frac{m_L}{m}\sigma^2_L - \frac{m_R}{m}\sigma^2_R,
$$

где $\sigma^2_{\text{node}} = \frac{1}{m_{\text{node}}} \sum_{i \in \text{node}} (y_i - \bar{y}_{\text{node}})^2$ — дисперсия отклика в узле. Иными словами, на каждом шаге алгоритм выбирает то расщепление, которое сильнее всего уменьшает разброс целевой переменной.

Для непрерывного признака $j$ перебор всех возможных порогов осуществляется сортировкой его уникальных значений в узле, после чего пороги выбираются как средние точки между соседними отсортированными значениями. Такой подход гарантирует, что каждое возможное разбиение разделяет данные по‑разному, и имеет вычислительную сложность $O(m \log m)$ для одного признака.


### 3. Критерии impurity для классификации

В отличие от регрессии, где качество разбиения измерялось уменьшением дисперсии, в задаче классификации с $K$ классами требуется количественно оценить, насколько «перемешаны» объекты разных классов в узле. Эту роль выполняют **impurity‑функции** (меры неоднородности). Пусть целевая переменная принимает значения $y_i \in \{1,\dots,K\}$, а в узле $t$, содержащем $m$ наблюдений, оценивается распределение классов

$$
\hat{p}_k = \frac{m_k}{m}, \qquad m_k = |\{i \in t: y_i = k\}|.
$$

Impurity-функция $I(\hat{p}_1,\dots,\hat{p}_K)$ должна удовлетворять трём естественным требованиям:
- $I \ge 0$, причём $I = 0$ тогда и только тогда, когда все объекты узла принадлежат одному классу;
- $I$ достигает максимума при равномерном распределении $\hat{p}_k = 1/K$;
- $I$ — симметричная и вогнутая функция вектора вероятностей, что гарантирует невозрастание impurity при расщеплении.

Стандартными мерами impurity являются ошибка классификации, энтропия и индекс Джини.

**Ошибка классификации**

$$
I_{\text{err}} = 1 - \max_k \hat{p}_k.
$$

Она непосредственно равна доле неверных предсказаний, если в узле предсказывается наиболее частый класс. Несмотря на интуитивную понятность, ошибка классификации обладает недостатком: она нечувствительна к изменениям в распределении классов, пока сохраняется наиболее частый класс. Например, узлы с распределениями $(0.6, 0.4)$ и $(0.9, 0.1)$ имеют одинаковую ошибку $0.4$ при предсказании класса $1$, хотя второй узел значительно «чище». Это свойство затрудняет выбор хороших разбиений, особенно на ранних этапах построения дерева.

**Энтропия**

$$
I_{\text{ent}} = -\sum_{k=1}^{K} \hat{p}_k \log_2 \hat{p}_k.
$$

Энтропия измеряет среднее количество информации (в битах), необходимое для описания класса случайно выбранного объекта из узла; нулевая энтропия означает полную определённость. Энтропия является строго вогнутой функцией и гораздо более чувствительна к изменениям состава узла: она принимает максимальное значение $\log_2 K$ при равномерном распределении и гладко спадает до нуля, когда один из классов начинает доминировать. Благодаря этому свойству энтропия поощряет разбиения, приводящие к резкому уменьшению неопределённости.

**Индекс Джини (Gini impurity)**

$$
I_{\text{Gini}} = \sum_{k=1}^{K} \hat{p}_k (1 - \hat{p}_k) = 1 - \sum_{k=1}^{K} \hat{p}_k^2.
$$

Его можно интерпретировать как вероятность ошибки, если классифицировать объект случайным образом, выбирая класс с вероятностями $\hat{p}_k$ (или, эквивалентно, вероятность того, что два случайно выбранных объекта узла принадлежат разным классам). Как и энтропия, индекс Джини строго вогнут и гладко изменяется от $1 - 1/K$ при равномерном распределении до $0$ в чистом узле. На практике он ведёт себя схоже с энтропией, но вычислительно несколько проще, так как не требует вычисления логарифмов.

Все три функции вогнуты по $\hat{\mathbf{p}}$ (энтропия и индекс Джини — строго вогнуты), достигают минимума $0$ на вырожденных распределениях и максимума при равномерном распределении.

Чтобы наглядно сравнить поведение impurity-мер, рассмотрим бинарный случай ($K=2$) и построим графики $I_{\text{err}}$, $I_{\text{ent}}$ и $I_{\text{Gini}}$ как функции $\hat{p}_1$ (при $\hat{p}_2 = 1 - \hat{p}_1$).




In [ ]:
# @title Сравнение impurity-мер в бинарном случае
import numpy as np
import matplotlib.pyplot as plt

p1 = np.linspace(0, 1, 200)
p2 = 1 - p1

I_err = 1 - np.maximum(p1, p2)
I_ent = - (p1 * np.log2(p1 + 1e-15) + p2 * np.log2(p2 + 1e-15))
I_gini = 1 - (p1**2 + p2**2)

plt.figure(figsize=(8,5))
plt.plot(p1, I_err, 'r-', label='Ошибка классификации', linewidth=2)
plt.plot(p1, I_ent, 'b-', label='Энтропия', linewidth=2)
plt.plot(p1, I_gini, 'g-', label='Индекс Джини', linewidth=2)
plt.xlabel(r'$\hat{p}_1$')
plt.ylabel('Impurity')
plt.title('Сравнение impurity-мер в бинарном случае')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Из графика видно, что ошибка классификации линейна по $\hat{p}_1$ и имеет излом в точке максимума, тогда как энтропия и индекс Джини являются гладкими вогнутыми функциями, плавно изменяющимися во всём диапазоне. Именно эта гладкость делает энтропию и Джини более чувствительными к изменениям состава узла: они сильнее «вознаграждают» за приближение к однородности, чем ошибка классификации. На практике это приводит к тому, что деревья, построенные с использованием энтропии или индекса Джини, склонны выбирать более сбалансированные и информативные разбиения, избегая локально оптимальных, но глобально бесполезных решений.

Расщепление узла выбирается так, чтобы максимизировать **информационный выигрыш (Information Gain)** — взвешенное уменьшение impurity:

$$
\text{Gain} = I_{\text{parent}} - \frac{m_L}{m} I_L - \frac{m_R}{m} I_R.
$$

Максимизация Gain эквивалентна минимизации средневзвешенной impurity дочерних узлов, что заставляет листья быть как можно более «чистыми».

Ниже приведена визуализация того, как решающее дерево разбивает двумерное пространство признаков на гиперпрямоугольники. Запустите ячейку, чтобы увидеть, как дерево глубины 3 разделяет данные «две луны» на области, соответствующие разным классам.




In [ ]:
# @title Визуализация решающего дерева на плоскости
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier

# Данные
X, y = make_moons(n_samples=200, noise=0.25, random_state=42)

# Обучаем дерево с ограниченной глубиной для наглядности
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

# Сетка для визуализации
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
Z = tree.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Визуализация
plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
plt.scatter(X[:,0], X[:,1], c=y, edgecolors='k', cmap='viridis')
plt.title('Разбиение пространства решающим деревом (глубина 3)')
plt.xlabel('Признак 1')
plt.ylabel('Признак 2')
plt.show()


#### Углублённый взгляд: связь impurity с вероятностными моделями и байесовским риском

Минимизация энтропии в дочерних узлах эквивалентна максимизации **взаимной информации** между признаками и классом. Пусть $Y$ — случайная величина класса, а $S$ — индикатор попадания в левый или правый дочерний узел. Тогда условная энтропия $H(Y \mid S)$ есть в точности взвешенная сумма энтропий $\frac{m_L}{m} H(Y \mid S=L) + \frac{m_R}{m} H(Y \mid S=R)$. Поскольку безусловная энтропия $H(Y)$ (энтропия родительского узла) фиксирована, уменьшение энтропии равно взаимной информации $I(Y; S)$ — мере того, насколько знание расщепления снижает неопределённость о классе.

Индекс Джини допускает аналогичную интерпретацию в терминах среднеквадратичной ошибки вероятностного прогноза. Пусть истинная принадлежность к классу кодируется one-hot вектором $\mathbf{y} \in \{0,1\}^K$, а прогнозные вероятности в узле — $\hat{\mathbf{p}}$. Тогда квадрат евклидовой ошибки составляет $\|\mathbf{y} - \hat{\mathbf{p}}\|^2 = \sum_{k=1}^{K} (y_k - \hat{p}_k)^2$. Математическое ожидание этой величины по истинному распределению классов $\hat{\mathbf{p}}$ даёт

$$
\mathbb{E}\bigl[ \|\mathbf{y} - \hat{\mathbf{p}}\|^2 \bigr]
= \sum_{j=1}^{K} \hat{p}_j \Bigl( (1 - \hat{p}_j)^2 + \sum_{k \ne j} \hat{p}_k^2 \Bigr)
= 1 - \sum_{k=1}^{K} \hat{p}_k^2 = I_{\text{Gini}}.
$$

Таким образом, индекс Джини есть ожидаемая квадратичная ошибка (мультиклассовый Brier score) внутри узла при прогнозе вероятностей классов, равных относительным частотам. Жадное уменьшение индекса Джини, следовательно, соответствует минимизации среднеквадратичной ошибки предсказанных вероятностей классов.

Байесовский риск (ожидаемая вероятность ошибки) при оптимальном байесовском классификаторе в узле равен $1 - \max_k \hat{p}_k$, то есть $I_{\text{err}}$. Энтропия и индекс Джини являются вогнутыми верхними границами для этой величины, поэтому их минимизация косвенно снижает и долю неверных классификаций. При этом они лучше различают «почти чистые» и «совершенно чистые» узлы, чем сама ошибка классификации, что делает процесс обучения более стабильным и чувствительным к улучшениям.

### 4. Почему impurity-меры предпочитают многозначные признаки?

И энтропия, и индекс Джини обладают скрытым смещением в пользу признаков с большим числом уникальных значений. Рассмотрим крайний случай: признак «уникальный идентификатор объекта» позволяет построить расщепление, при котором каждый дочерний узел содержит ровно одно наблюдение. Тогда impurity листьев обращается в нуль, информационный выигрыш достигает максимума, но такое расщепление абсолютно бесполезно для обобщения.

Для компенсации этого эффекта алгоритм C4.5 вместо обычного Information Gain использует **Information Gain Ratio** — информационный выигрыш, нормированный на собственную энтропию расщепления (Split Information):

$$
\text{GainRatio} = \frac{\text{Gain}}{-\sum_{v} \frac{m_v}{m} \log_2 \frac{m_v}{m}},
$$

где сумма берётся по всем дочерним узлам $v$. Знаменатель — это энтропия самого разбиения: он измеряет, насколько сильно расщепление само по себе сегментирует данные. Чем больше уникальных значений у признака, тем выше эта энтропия, и GainRatio штрафует такие признаки, отдавая предпочтение более сбалансированным разбиениям. В алгоритме CART, обычно применяемом в современных реализациях, эта проблема менее выражена благодаря бинарной природе расщеплений; для категориальных признаков также используют специальные методы объединения категорий, что дополнительно снижает неоправданное дробление.

### 5. Проблема переобучения и регуляризация

Решающее дерево, не имеющее ограничений на рост, способно идеально воспроизвести обучающую выборку — достаточно построить дерево, в котором каждый лист содержит ровно один объект. При этом обучающая ошибка станет нулевой, однако дисперсия модели окажется чрезвычайно высокой, и качество на новых данных резко ухудшится.

Для борьбы с переобучением применяют стратегии регуляризации, разделяемые на **pre‑pruning** (раннюю остановку) и **post‑pruning** (отсечение ветвей после построения полного дерева).

**Pre‑pruning** вводит критерии, запрещающие дальнейшее разбиение узла, если:
- число объектов в узле меньше `min_samples_split`;
- хотя бы один из дочерних узлов получит меньше `min_samples_leaf` объектов;
- достигнута максимальная глубина `max_depth`;
- улучшение impurity меньше заданного порога `min_impurity_decrease`.

Эти параметры непосредственно ограничивают сложность дерева.

**Cost‑complexity pruning** (используемый в оригинальном алгоритме CART) строит последовательность вложенных поддеревьев, минимизируя штрафную функцию:

$$
R_\alpha(T) = R(T) + \alpha |T|,
$$

где $R(T)$ — суммарная ошибка дерева на обучающей выборке (например, MSE для регрессии или число неверных классификаций для классификации), $|T|$ — количество листьев, а $\alpha \ge 0$ — параметр, балансирующий точность и сложность. При $\alpha = 0$ оптимальным является самое глубокое дерево; с ростом $\alpha$ более выгодными становятся поддеревья меньшего размера. Алгоритм эффективно находит всю последовательность оптимальных поддеревьев для всех возможных значений $\alpha$ путём последовательного отсечения наименее «важных» ветвей. Оптимальное значение $\alpha$ выбирается с помощью кросс‑валидации, что даёт дерево с хорошим балансом между смещением и дисперсией.


### Вопросы для самопроверки

#### Для начинающих
1. Что такое impurity-мера в контексте решающих деревьев и каким требованиям она должна удовлетворять?
2. Почему в качестве критерия расщепления при классификации предпочтительнее использовать энтропию или индекс Джини, а не непосредственно ошибку классификации?
3. Зачем нужна регуляризация (pruning) деревьев? В чём разница между pre‑pruning и post‑pruning?
4. Как работает критерий «минимальное число объектов в листе» (`min_samples_leaf`) и к каким последствиям для структуры дерева он приводит?
5. Что такое Information Gain Ratio и для чего он был введён? В каком алгоритме он применяется?

#### Для профессионалов
1. Докажите строгую вогнутость энтропии и индекса Джини как функций вектора вероятностей $\hat{\mathbf{p}}$ и покажите, что они достигают глобального максимума при равномерном распределении.
2. Объясните, почему индекс Джини численно равен ожидаемой ошибке классификации, если решение о классе принимается случайным выбором по распределению $\hat{\mathbf{p}}$. Как это связано с Brier score для мультиклассового прогноза вероятностей?
3. Выведите градиент энтропии по предсказанным вероятностям классов в листе $\partial I_{\text{ent}} / \partial \hat{p}_k$ и проинтерпретируйте его в контексте того, как изменение частоты класса влияет на impurity.
4. Обсудите NP‑полноту задачи построения оптимального (минимального по числу листьев или ошибке) решающего дерева для заданной обучающей выборки. Какие следствия это имеет для алгоритмов обучения деревьев на практике?

## Реализация решающего дерева с нуля и анализ сложности

Теоретические принципы, изложенные в предыдущем разделе, допускают непосредственную алгоритмическую реализацию. Ниже мы создадим с нуля регрессионное и классификационное решающие деревья, руководствуясь жадной стратегией CART, и проанализируем их вычислительные характеристики.

Запустите первую кодовую ячейку — она содержит определение узла дерева и полные реализации классов `DecisionTreeRegressor` и `DecisionTreeClassifier`.

In [ ]:
# @title Реализация решающих деревьев (регрессия и классификация)
import numpy as np

class Node:
    def __init__(self, feature_index=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# ============================================================
# Регрессионное дерево
# ============================================================
class DecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]
        if (self.max_depth is not None and depth >= self.max_depth) \
           or n_samples < self.min_samples_split \
           or np.all(y == y[0]):
            return Node(value=np.mean(y))

        best_mse = float('inf')
        best_feature, best_threshold = None, None
        best_left_idx, best_right_idx = None, None

        for j in range(self.n_features_):
            sorted_idx = np.argsort(X[:, j])
            X_sorted = X[sorted_idx, j]
            y_sorted = y[sorted_idx]
            thresholds = (X_sorted[:-1] + X_sorted[1:]) / 2.0

            for i, th in enumerate(thresholds):
                left_idx = sorted_idx[:i+1]
                right_idx = sorted_idx[i+1:]
                n_left = i + 1
                n_right = n_samples - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                mse_left = np.var(y_sorted[:i+1]) * n_left
                mse_right = np.var(y_sorted[i+1:]) * n_right
                total_mse = mse_left + mse_right

                if total_mse < best_mse:
                    best_mse = total_mse
                    best_feature = j
                    best_threshold = th
                    best_left_idx = left_idx
                    best_right_idx = right_idx

        if best_feature is None:
            return Node(value=np.mean(y))

        left_subtree = self._build_tree(X[best_left_idx], y[best_left_idx], depth+1)
        right_subtree = self._build_tree(X[best_right_idx], y[best_right_idx], depth+1)
        return Node(feature_index=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

# ============================================================
# Классификационное дерево
# ============================================================
class DecisionTreeClassifier:
    def __init__(self, criterion='gini', max_depth=None,
                 min_samples_split=2, min_samples_leaf=1):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _impurity(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y, minlength=self.n_classes_)
        p = counts / len(y)
        if self.criterion == 'gini':
            return 1 - np.sum(p ** 2)
        elif self.criterion == 'entropy':
            p = p[p > 0]
            return -np.sum(p * np.log2(p))
        else:
            raise ValueError("criterion must be 'gini' or 'entropy'")

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]
        if (self.max_depth is not None and depth >= self.max_depth) \
           or n_samples < self.min_samples_split \
           or len(np.unique(y)) == 1:
            counts = np.bincount(y, minlength=self.n_classes_)
            return Node(value=np.argmax(counts))

        parent_impurity = self._impurity(y)
        best_gain = 0.0
        best_feature, best_threshold = None, None
        best_left_idx, best_right_idx = None, None

        for j in range(self.n_features_):
            sorted_idx = np.argsort(X[:, j])
            X_sorted = X[sorted_idx, j]
            y_sorted = y[sorted_idx]
            thresholds = (X_sorted[:-1] + X_sorted[1:]) / 2.0

            for i, th in enumerate(thresholds):
                left_idx = sorted_idx[:i+1]
                right_idx = sorted_idx[i+1:]
                n_left = i + 1
                n_right = n_samples - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                left_impurity = self._impurity(y_sorted[:i+1])
                right_impurity = self._impurity(y_sorted[i+1:])
                weighted_imp = (n_left * left_impurity + n_right * right_impurity) / n_samples
                gain = parent_impurity - weighted_imp

                if gain > best_gain:
                    best_gain = gain
                    best_feature = j
                    best_threshold = th
                    best_left_idx = left_idx
                    best_right_idx = right_idx

        if best_feature is None:
            counts = np.bincount(y, minlength=self.n_classes_)
            return Node(value=np.argmax(counts))

        left_subtree = self._build_tree(X[best_left_idx], y[best_left_idx], depth+1)
        right_subtree = self._build_tree(X[best_right_idx], y[best_right_idx], depth+1)
        return Node(feature_index=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

print("Классы DecisionTreeRegressor и DecisionTreeClassifier загружены.")

#### Проверка на синтетических данных: регрессия

Сгенерируем данные $y = x^2 + \sin(5x) + \varepsilon$, обучим наше регрессионное дерево с различной глубиной и сравним его с `sklearn.tree.DecisionTreeRegressor`. Запустите ячейку ниже — она построит графики для трёх значений `max_depth`.

In [ ]:
# @title Регрессия: сравнение нашей реализации с sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor as SKDecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Генерация данных
np.random.seed(42)
n = 100
x = np.linspace(-2, 2, n)
y = x**2 + np.sin(5*x) + np.random.normal(0, 0.3, n)
X = x.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

depths = [2, 5, 10]
plt.figure(figsize=(15, 5))
for i, d in enumerate(depths):
    our_tree = DecisionTreeRegressor(max_depth=d)
    our_tree.fit(X_train, y_train)
    y_pred_our = our_tree.predict(X_test)

    sk_tree = SKDecisionTreeRegressor(max_depth=d, random_state=0)
    sk_tree.fit(X_train, y_train)
    y_pred_sk = sk_tree.predict(X_test)

    x_plot = np.linspace(-2.2, 2.2, 200).reshape(-1,1)
    plt.subplot(1, 3, i+1)
    plt.scatter(X_train, y_train, alpha=0.5, label='train')
    plt.plot(x_plot, our_tree.predict(x_plot), 'r-', label='our')
    plt.plot(x_plot, sk_tree.predict(x_plot), 'g--', label='sklearn')
    plt.title(f'max_depth={d}')
    plt.legend()
plt.tight_layout()
plt.show()

#### Проверка на синтетических данных: классификация

Используем датасет «две луны» (`make_moons`). Обучим наш классификатор с разной глубиной и построим контурные графики границы решений. Запустите ячейку, чтобы увидеть, как с ростом глубины граница становится всё более изрезанной.

In [ ]:
# @title Классификация: границы решений при разной глубине
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=200, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

plt.figure(figsize=(15, 5))
for i, d in enumerate([1, 3, None]):
    tree = DecisionTreeClassifier(max_depth=d)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                           np.linspace(x2_min, x2_max, 200))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    Z = tree.predict(grid).reshape(xx1.shape)

    plt.subplot(1, 3, i+1)
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', cmap='RdYlBu')
    title = f'max_depth={d}' if d is not None else 'без ограничения'
    plt.title(f'{title} (acc={acc:.2f})')
plt.tight_layout()
plt.show()

#### Анализ переобучения и роль гиперпараметров

Построим кривые обучения при изменении `max_depth` для регрессионного датасета. Это наглядно покажет компромисс между смещением и дисперсией.

In [ ]:
# @title Кривые обучения: переобучение при увеличении глубины
max_depths = np.arange(1, 21)
train_mse, test_mse = [], []
for d in max_depths:
    tree = DecisionTreeRegressor(max_depth=d)
    tree.fit(X_train, y_train)
    train_mse.append(mean_squared_error(y_train, tree.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, tree.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(max_depths, train_mse, 'bo-', label='train MSE')
plt.plot(max_depths, test_mse, 'ro-', label='test MSE')
plt.xlabel('max_depth')
plt.ylabel('MSE')
plt.legend()
plt.grid()
plt.title('Переобучение решающего дерева')
plt.show()

Визуализация подтверждает совпадение предсказаний нашей и библиотечной моделей. Дерево малой глубины даёт грубую кусочно‑постоянную аппроксимацию, с увеличением глубины детализация возрастает, но появляется риск переобучения. Другие гиперпараметры (`min_samples_leaf`, `min_samples_split`) аналогично ограничивают рост дерева и предотвращают излишнюю подгонку под шум.



#### Вопросы для самопроверки

**Для начинающих**
1. Как в коде осуществляется выбор порога для расщепления? Почему пороги берутся как середины между соседними уникальными значениями признака?
2. Почему дерево без ограничений (max_depth=None, min_samples_leaf=1) практически всегда переобучается?
3. Каким образом гиперпараметр `min_samples_leaf` влияет на сложность дерева и его способность к обобщению?
4. Как визуализировать границу решений классификатора и что означает её сильная изрезанность?

**Для профессионалов**
1. Объясните, почему сложность наивного перебора порогов составляет $O(D \, m \log m)$ на один узел, и предложите способы её снижения с помощью предварительной сортировки признаков один раз для всего обучения, а не на каждом узле.
2. Сравните жадный алгоритм CART с поиском глобально оптимального дерева минимального размера. Почему последняя задача NP‑полна? К каким практическим последствиям это приводит?
3. При наличии повторяющихся значений признака выбор порога как середины может приводить к вырожденным ситуациям (например, порог равен значению признака). Как обеспечить численную устойчивость и корректность разбиения в таких случаях?